### 🌀 Ring Sliding Window Attention in PyTorch (Conceptual Implementation)

Ring Sliding Window Attention is mainly used in **distributed GPU settings** where each GPU holds part of the sequence and **Key/Value tensors rotate between GPUs in a ring**.

Below is a **simplified conceptual PyTorch implementation** to help you understand the mechanics.
(This is educational code — real systems like Megatron-LM or DeepSpeed implement it inside optimized CUDA kernels.)

---

# 🧠 Idea of the Algorithm

Each GPU:

1️⃣ Computes **local Q, K, V**
2️⃣ Computes **local sliding window attention**
3️⃣ Sends its **K,V to next GPU**
4️⃣ Receives **K,V from previous GPU**
5️⃣ Computes attention again
6️⃣ Repeat until window coverage is satisfied

Communication happens in a **ring topology**.

```
GPU0 → GPU1 → GPU2 → GPU3 → GPU0
```
---

# ⚙️ Important Components Explained

### 1️⃣ Multi-head projection

```
Q = XWq
K = XWk
V = XWv
```

$$
Q = XW_Q,\quad K = XW_K,\quad V = XW_V
$$

---

### 2️⃣ Sliding Window Mask

Allowed tokens:

$$
j \in [i-W+1 , i]
$$

Where **W = window size**.

---

### 3️⃣ Attention Computation

$$
Attention(Q,K,V) =
softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

---

### 4️⃣ Ring Communication

Each iteration:

```
send KV → next GPU
receive KV ← previous GPU
```

Topology:

```
GPU0 → GPU1 → GPU2 → GPU3 → GPU0
```

---

# 📊 Complexity

Compute per GPU:

$$
O\left(\frac{N}{G} \times W\right)
$$

Memory per GPU:

$$
O(N/G)
$$

Where:

* **N** = sequence length
* **G** = number of GPUs
* **W** = sliding window size

---

# 🚀 Why Real Implementations Are Different

Production systems improve this with:

* **FlashAttention kernels**
* **NCCL ring communication**
* **fused attention kernels**
* **asynchronous KV rotation**

Libraries implementing optimized versions include:

* FlashAttention
* Megatron-LM
* DeepSpeed

---

# ⚠️ Important Note

This code demonstrates the **concept**, but in real models:

* KV rotation is **asynchronous**
* masking is **block sparse**
* attention is computed using **GPU kernels**

Otherwise it would be **too slow in Python**.

---

✅ If you'd like, I can also show you:

* **A much cleaner PyTorch implementation used in research papers**
* **How Mistral implements sliding window attention internally**
* **How FlashAttention + Ring Attention work together**

Those tricks are what make **modern LLMs extremely fast**.


In [1]:
import torch
import torch.nn as nn
import torch.distributed as dist
import math


class RingSlidingWindowAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, window_size):
        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)

        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):

        B, T, C = x.shape

        # Project QKV
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # reshape to heads
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1,2)

        # attention output accumulator
        output = torch.zeros_like(q)

        world_size = dist.get_world_size()
        rank = dist.get_rank()

        next_rank = (rank + 1) % world_size
        prev_rank = (rank - 1 + world_size) % world_size

        k_curr = k
        v_curr = v

        # number of ring passes needed
        for _ in range(world_size):

            # sliding window attention
            scores = torch.matmul(q, k_curr.transpose(-2,-1))
            scores = scores / math.sqrt(self.head_dim)

            # sliding window mask
            mask = torch.ones(T, T, device=x.device)
            mask = torch.triu(mask, diagonal=self.window_size)
            scores = scores.masked_fill(mask.bool(), float('-inf'))

            attn = torch.softmax(scores, dim=-1)

            out = torch.matmul(attn, v_curr)

            output += out

            # send KV to next GPU
            dist.send(k_curr, dst=next_rank)
            dist.send(v_curr, dst=next_rank)

            # receive KV from previous GPU
            k_recv = torch.zeros_like(k_curr)
            v_recv = torch.zeros_like(v_curr)

            dist.recv(k_recv, src=prev_rank)
            dist.recv(v_recv, src=prev_rank)

            k_curr = k_recv
            v_curr = v_recv

        # merge heads
        output = output.transpose(1,2).contiguous().view(B, T, C)

        return self.out_proj(output)